# CodeDoctor — AppsRetrieval evaluation on a hosted GPU

Runs the two-pass retrieval pipeline through `mteb.evaluate` and writes the
submission artifact. The pipeline source is embedded below, generated from
the repository by `scripts/make_notebook.py` — there is no stale bundle to
drift out of sync.

The *solution* is CPU-only; a GPU is used here purely to encode 17,530 texts
once. Everything downstream runs on cached vectors.


In [ ]:
# 1. Environment
import torch

print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    major = torch.cuda.get_device_capability(0)[0]
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'{name}  compute {major}.x  {mem:.1f} GB')
    # Turing (T4, compute 7.5) has no native bfloat16: torch emulates it and
    # the encode slows by roughly 10x. select_dtype handles this for us.
else:
    print('No GPU. This will work but the encode takes hours.')


In [ ]:
# 2. Unpack the pipeline (generated from the repo working tree)
PAYLOAD = "..."

import base64, io, os, sys, tarfile

WORK = '/content/codedoctor'
os.makedirs(WORK, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(PAYLOAD)), mode='r:gz') as tar:
    tar.extractall(WORK)
sys.path.insert(0, WORK)
os.chdir(WORK)
print(sorted(os.listdir(WORK)))


In [ ]:
# 3. Dependencies (mteb pinned: it decides the result-JSON schema)
!pip install -q -U 'mteb==2.21.0' 'sentence-transformers>=5.0' 'transformers>=4.57' datasets pyyaml


## Encode once

Dumps texts, qrels and embeddings to `artifacts/apps`. Reranker tuning then
runs on those cached vectors in seconds instead of hours.


In [ ]:
!python scripts/encode_apps.py --tier interactive --device auto --tag f2llm06b
!python scripts/encode_apps.py --tier accuracy    --device auto --tag f2llm17b


## Fit the second pass on TRAIN, report on held-out DEV

The dev split is seeded-random and filtered to the query family the test
split is actually made of — see `app/pipeline/splits.py` for why a
contiguous half-split of this corpus is not exchangeable.


In [ ]:
!python scripts/tune_reranker.py --model f2llm17b --depth 50


## Stage-by-stage ablation, each gain gated on a paired bootstrap


In [ ]:
!python scripts/dev_eval.py --model f2llm17b
!python scripts/dev_eval.py --model f2llm17b --execution


## The submission artifact

Produced by `mteb.evaluate` itself — no score is assembled by hand.


In [ ]:
!python scripts/evaluate_mteb.py --device auto --predictions build/predictions
!python scripts/validate_submission.py appsretrieval_results.json


In [ ]:
# Download the artifacts
from google.colab import files  # noqa: F401
for name in ('appsretrieval_results.json', 'submission_predictions.csv',
             'submission_metrics.csv', 'config/reranker.json'):
    if os.path.exists(name):
        files.download(name)
